# A random walk, its statistics and its figures

`Stats` holds a seeded stream (xoshiro256**, the same draws on every
machine), `Arrays` the array operations, `Plot` the figures. The state
cell below draws a thousand standard normal steps and sums them; the
cells after it read `Walk.steps` and `Walk.pos` by name.

In [ ]:
STATEFUL DEFINITION MODULE Walk ;

VAR steps, pos : SLICE OF F64 ;

END Walk.

IMPLEMENTATION MODULE Walk ;

IMPORT Arrays ;
IMPORT Io ;
IMPORT Stats ;

VAR
  st : Stats.Stream ;
  i : I64 ;

BEGIN
  st := Stats.Seed (42) ;
  steps := NEW (F64, 1000) ;
  FOR i := 0 TO 999 DO steps[i] := Stats.Normal (st) END ;
  pos := Arrays.CumSum (steps)
EXCEPT
| ValueRange :
    Io.ErrLine ('the stream failed')
END Walk.

In [ ]:
Stats.Mean (Walk.steps)

In [ ]:
Stats.Std (Walk.steps)

In [ ]:
Walk.pos

A histogram of the steps: `Stats.Histogram` counts into equal bins,
`Stats.BinEdges` says where they are, and a bar series is drawn at
the bin centres.

In [ ]:
MODULE Hist ;

IMPORT Faults ;
IMPORT Io ;
IMPORT Plot ;
IMPORT Stats ;
IMPORT Walk ;

CONST Bins = 24 ;

VAR
  counts : SLICE OF I64 ;
  edges, at, v : SLICE OF F64 ;
  i : I64 ;

BEGIN
  counts := Stats.Histogram (Walk.steps, Bins, -4.0, 4.0) ;
  edges := Stats.BinEdges (Bins, -4.0, 4.0) ;
  at := NEW (F64, Bins) ;
  v := NEW (F64, Bins) ;
  FOR i := 0 TO Bins - 1 DO
    at[i] := 0.5 * (edges[i] + edges[i + 1]) ;
    v[i] := F64 (counts[i])
  END ;
  Plot.ClearFigure () ;
  Plot.AddBars (at, v, 0, 'steps') ;
  Io.WriteFile ('steps.svg', Plot.Render ('1000 standard normal steps', 'step', 'count'))
EXCEPT
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| ValueRange :
    Io.ErrLine ('a value out of range')
| IndexError :
    Io.ErrLine ('out of range')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END Hist.

The walk itself, with a rolling mean over fifty steps laid over it.
`Stats.RollingMean` answers `LEN - window + 1` values, so the smooth
line starts at the window's end.

In [ ]:
MODULE Smooth ;

IMPORT Faults ;
IMPORT Io ;
IMPORT Plot ;
IMPORT Stats ;
IMPORT Walk ;

CONST Window = 50 ;

VAR
  t, ts, sm : SLICE OF F64 ;
  i, n : I64 ;

BEGIN
  n := LEN (Walk.pos) ;
  t := NEW (F64, n) ;
  FOR i := 0 TO n - 1 DO t[i] := F64 (i) END ;
  sm := Stats.RollingMean (Walk.pos, Window) ;
  ts := SLICE (t, Window - 1, n - Window + 1) ;
  Plot.ClearFigure () ;
  Plot.AddLine (t, Walk.pos, 0, 'position') ;
  Plot.AddLine (ts, sm, 1, 'rolling mean, 50 steps') ;
  Io.WriteFile ('walk.svg', Plot.Render ('a random walk', 'step', 'position'))
EXCEPT
| Stats.TooFew (got, want) :
    Io.ErrLine ('too few values')
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| ValueRange :
    Io.ErrLine ('a value out of range')
| IndexError :
    Io.ErrLine ('out of range')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END Smooth.

Values also travel between programs by name, through `NbCells`: one
cell stores, a later one reads. The files are Parquet in
`02-random-walk.m9data` beside the notebook, so pandas reads them
too: `pd.read_parquet('02-random-walk.m9data/walk.f64s.parquet')`.

In [ ]:
MODULE Store ;

IMPORT Faults ;
IMPORT Io ;
IMPORT NbCells ;
IMPORT Walk ;

BEGIN
  NbCells.PutF64s ('walk', Walk.pos)
EXCEPT
| Faults.BadArg (what) :
    Io.ErrLine ('not a name: ' + what)
| Faults.SizeError (got, want) :
    Io.ErrLine ('a size')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
| ValueRange :
    Io.ErrLine ('a value out of range')
| Overflow :
    Io.ErrLine ('overflow')
| IndexError :
    Io.ErrLine ('out of range')
END Store.

In [ ]:
MODULE Quantiles ;

IMPORT Faults ;
IMPORT Fmt ;
IMPORT Io ;
IMPORT NbCells ;
IMPORT Stats ;

VAR
  pool : POOL ;
  xs : SLICE OF F64 ;

BEGIN
  xs := NbCells.GetF64s (pool, 'walk') ;
  Io.WriteLine ('positions: ' + Fmt.I64Str (LEN (xs))) ;
  Io.WriteLine ('   5th percentile ' + Fmt.Fixed (Stats.Percentile (xs, 5.0), 2)) ;
  Io.WriteLine ('   median         ' + Fmt.Fixed (Stats.Median (xs), 2)) ;
  Io.WriteLine ('   95th percentile ' + Fmt.Fixed (Stats.Percentile (xs, 95.0), 2))
EXCEPT
| NbCells.Missing (name) :
    Io.ErrLine ('nothing stored as ' + name + ': run the cell above first')
| NbCells.WrongType (name, stored, wanted) :
    Io.ErrLine (name + ' holds ' + stored + ', not ' + wanted)
| Stats.TooFew (got, want) :
    Io.ErrLine ('too few values')
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| Io.IOError (path) :
    Io.ErrLine ('cannot read ' + path)
| ValueRange :
    Io.ErrLine ('a value out of range')
| Overflow :
    Io.ErrLine ('overflow')
| IndexError :
    Io.ErrLine ('out of range')
END Quantiles.